# Stage 9a — What is actually inside a GGUF

**Goal:** open the file format your model now lives in, and see that it is
genuinely self-contained.

This notebook runs on your laptop with **no PyTorch**. The `gguf` package is
pure Python plus numpy — it reads the format directly. That is only possible
because GGUF was designed for exactly this: a single file that a C++ binary (or
a small Python library) can memory-map and run, with no framework, no config
files sitting beside it, and no tokenizer to load separately.

### The format, briefly

```
┌──────────────────────────────────────────┐
│ magic "GGUF" + version                    │
├──────────────────────────────────────────┤
│ metadata key/value pairs                  │  architecture, hyperparameters,
│   general.architecture = "llama"          │  the entire tokenizer vocabulary,
│   llama.block_count = 8                   │  and the chat template
│   tokenizer.ggml.tokens = [...]           │
│   tokenizer.chat_template = "..."         │
├──────────────────────────────────────────┤
│ tensor info: name, shape, dtype, offset   │
├──────────────────────────────────────────┤
│ tensor data (aligned, mmap-friendly)      │
└──────────────────────────────────────────┘
```

Contrast that with the HF repo from stage 7, where the weights, the config, and
the tokenizer were three separate files that had to agree with each other.

In [ ]:
# --- Local bootstrap -------------------------------------------------------
# Kernel: "tinyllm (local, no torch)" -- registered by scripts\setup_local.ps1.
# There is deliberately no PyTorch in this environment.
import subprocess, sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from tinyllm import config
from tinyllm.config import model_cfg, quant_cfg, serve_cfg, gen_cfg, hub

MODELS = REPO / "models"
VENDOR = REPO / "vendor" / "llamacpp"

def llama_tool(name):
    """Locate a llama.cpp binary; its position inside the release zip moves."""
    hits = list(VENDOR.rglob(f"{name}.exe")) or list(VENDOR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {VENDOR}. Run scripts\\setup_local.ps1.")
    return hits[0]

import importlib.util
assert importlib.util.find_spec("torch") is None, (
    "torch is installed in the local venv -- it should not be. See pyproject.toml."
)
print(f"repo:   {REPO}")
print(f"models: {[p.name for p in sorted(MODELS.glob('*.gguf'))] or 'none yet'}")
print("torch:  absent, by design")

## 9a.1 — Open the file

In [ ]:
from gguf import GGUFReader
import numpy as np

path = MODELS / f"{config.PROJECT_NAME}-f16.gguf"
assert path.exists(), f"{path.name} not found. Run scripts\\pull_model.ps1 first."

reader = GGUFReader(str(path))
print(f"{path.name}  ({path.stat().st_size / 1e6:.1f} MB)")
print(f"  metadata fields: {len(reader.fields)}")
print(f"  tensors:         {len(reader.tensors)}")

## 9a.2 — The metadata

Everything llama.cpp needs to reconstruct the architecture. Compare these
against `config.py` — they should match exactly, because they were derived from
it seven stages ago.

In [ ]:
def show(field):
    try:
        v = field.contents()
    except Exception:
        return "<complex>"
    if isinstance(v, (list, tuple)) and len(v) > 6:
        return f"[{len(v)} items] {list(v[:4])}..."
    if isinstance(v, str) and len(v) > 70:
        return v[:70] + "..."
    return v

print("ARCHITECTURE")
for name, field in reader.fields.items():
    if name.startswith(("general.", "llama.")):
        print(f"  {name:<42} {show(field)}")

In [ ]:
# Check the file agrees with config.py -- a mismatch means something drifted
# between training and conversion.
def get(key, default=None):
    f = reader.fields.get(key)
    if f is None:
        return default
    try:
        return f.contents()
    except Exception:
        return default

checks = [
    ("block_count",      get("llama.block_count"),                 model_cfg.num_hidden_layers),
    ("embedding_length", get("llama.embedding_length"),            model_cfg.hidden_size),
    ("head_count",       get("llama.attention.head_count"),        model_cfg.num_attention_heads),
    ("head_count_kv",    get("llama.attention.head_count_kv"),     model_cfg.num_key_value_heads),
    ("feed_forward",     get("llama.feed_forward_length"),         model_cfg.intermediate_size),
    ("context_length",   get("llama.context_length"),              model_cfg.max_position_embeddings),
    ("rope_freq_base",   get("llama.rope.freq_base"),              model_cfg.rope_theta),
]

print(f"{'field':<20} {'in GGUF':>12} {'in config.py':>14}   match")
print("-" * 60)
ok = True
for name, got, want in checks:
    same = (got is not None) and (float(got) == float(want))
    ok &= same
    print(f"{name:<20} {str(got):>12} {str(want):>14}   {'yes' if same else 'NO'}")
assert ok, "GGUF metadata disagrees with config.py"
print("\nThe file describes exactly the model we designed in stage 3.")

## 9a.3 — The tokenizer travelled with it

This is the part that surprises people: the whole vocabulary is *inside* the
model file. No `tokenizer.model`, no `tokenizer.json`, nothing beside it.

In [ ]:
tokens = get("tokenizer.ggml.tokens", [])
scores = get("tokenizer.ggml.scores", [])
ttypes = get("tokenizer.ggml.token_type", [])

print(f"  model type   {get('tokenizer.ggml.model')}")
print(f"  vocab size   {len(tokens):,}")
print(f"  BOS / EOS    {get('tokenizer.ggml.bos_token_id')} / {get('tokenizer.ggml.eos_token_id')}")
print()
print("  first 12 tokens:")
for i in range(min(12, len(tokens))):
    t = tokens[i]
    t = t.decode("utf-8", "replace") if isinstance(t, bytes) else str(t)
    print(f"    {i:>5}  type={ttypes[i] if i < len(ttypes) else '?'}  {t!r}")

print("\n  some learned merges (ids 400-410):")
for i in range(400, min(410, len(tokens))):
    t = tokens[i]
    t = t.decode("utf-8", "replace") if isinstance(t, bytes) else str(t)
    print(f"    {i:>5}  {t!r}")

In [ ]:
# The chat template -- hop 4 of 4 for that string. config.py -> tokenizer_config
# -> GGUF -> llama-server. This is where the server reads it from.
tmpl = get("tokenizer.chat_template")
if tmpl:
    print("chat template found in the GGUF:\n")
    print(tmpl[:500])
else:
    print("WARNING: no chat template. llama-server would guess a format, and the")
    print("model would see prompts unlike anything it was trained on.")

## 9a.4 — The tensors

Every weight matrix, with the names llama.cpp expects. Note how they map onto
`model_scratch.py`: `blk.N.attn_q` is `layers[N].self_attn.q_proj`, and so on.

In [ ]:
rows = []
for t in reader.tensors:
    n = int(np.prod(t.shape))
    rows.append((t.name, tuple(int(x) for x in t.shape), str(t.tensor_type).split(".")[-1], n))

print(f"{'tensor':<32} {'shape':<18} {'dtype':<10} {'params':>12}")
print("-" * 76)
for name, shape, dt, n in rows[:14]:
    print(f"{name:<32} {str(shape):<18} {dt:<10} {n:>12,}")
print(f"... {len(rows) - 14} more")

total = sum(n for *_, n in rows)
print(f"\ntotal parameters in file: {total:,}")
print(f"config.py predicts:       {model_cfg.n_params:,}")

The count is lower than `config.py` predicts, and that is correct: **tied
embeddings**. `token_embd.weight` appears once but is used both to embed input
tokens and to produce output logits. The 3.1M-parameter saving from stage 3 is
visible right here in the file layout.

In [ ]:
# Group by role to see where the parameters live.
from collections import defaultdict

groups = defaultdict(int)
for name, shape, dt, n in rows:
    if "token_embd" in name:      key = "embedding (tied)"
    elif "attn" in name:          key = "attention"
    elif "ffn" in name:           key = "feed-forward"
    elif "norm" in name:          key = "normalization"
    else:                          key = "other"
    groups[key] += n

import matplotlib.pyplot as plt
labels = list(groups)
sizes = [groups[k] for k in labels]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(labels, sizes, color=["#4C78A8", "#F58518", "#54A24B", "#E45756", "#B279A2"])
ax.bar_label(bars, labels=[f"{s:,} ({s/total:.0%})" for s in sizes], padding=4, fontsize=9)
ax.set_xlabel("parameters"); ax.set_xlim(0, max(sizes) * 1.35)
ax.set_title("Where the parameters live")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 9a.5 — Look at an actual weight

The point of this cell is that there is no mystery left. These are the numbers
your training run produced, readable from disk with numpy.

In [ ]:
t = next(t for t in reader.tensors if t.name == "blk.0.attn_q.weight")
w = np.array(t.data, dtype=np.float32)

print(f"{t.name}  shape={tuple(int(x) for x in t.shape)}  dtype={t.tensor_type}")
print(f"  mean {w.mean():+.5f}   std {w.std():.5f}")
print(f"  min  {w.min():+.5f}   max {w.max():+.5f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(w.ravel(), bins=120, color="#4C78A8")
ax.set_title(f"{t.name} — weight distribution after training")
ax.set_xlabel("value"); ax.set_ylabel("count")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("\nRoughly zero-centred and bell-shaped. That matters for the next notebook:")
print("quantization maps this range onto a handful of levels, and a well-behaved")
print("distribution is what makes 4 bits per weight survivable at all.")

## Stage 9a gate

- [x] GGUF metadata matches `config.py`
- [x] Tokenizer vocabulary is inside the file
- [x] Chat template is inside the file
- [x] Tensor count reflects tied embeddings
- [x] All of it read without PyTorch

**Next:** `10_quant_tradeoff.ipynb`